# Looking at enriched pathways in individual cancers

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

INDIVIDUAL_ENRICHMENTS_DIR = "individual_cancers_enrichments"
INDIVIDUAL_ENRICHMENTS_FILE = f"urls_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
INDIVIDUAL_ENRICHMENTS_FILE_PATH = os.path.join(INDIVIDUAL_ENRICHMENTS_DIR, INDIVIDUAL_ENRICHMENTS_FILE)

if not os.path.isdir(INDIVIDUAL_ENRICHMENTS_DIR):
    os.mkdir(INDIVIDUAL_ENRICHMENTS_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# The ranked enrichment analysis service wants ranking scores
# where "bigger is better", such as expression values or
# t scores. We are ranking by adjusted p-values, where smaller
# is better. So, we'll create a column of (1 - adj_p) to use
# for ranking.
data = data.assign(one_minus_adj_p=1 - data["adj_p"])

# Make a column where all increases are +1 and all decreases 
# are -1, because these are ratioed abundances, so we can't 
# compare magnitudes between genes--we can only compare whether 
# there was a change, and whether it was positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [4]:
data.head()

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,one_minus_adj_p,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0000,0.00000,True,A1BG,1.00000,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0000,0.00000,True,A1CF,1.00000,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0000,0.00000,True,A2M,1.00000,1.0
3,ccrcc,"('A4GALT', 'NP_001304967.1')",0.479410,0.1735,0.21431,False,A4GALT,0.78569,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0000,0.00000,True,AAAS,1.00000,1.0


## Run enrichment analysis

In [5]:
# For each cancer, find enriched pathways based on p value for differential expression   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    
    ranked_data = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "one_minus_adj_p"]].\
        set_index("protein_str").\
        rename(columns={"one_minus_adj_p": f"{cancer_type}_one_minus_adj_p"})

    unneeded_token, cancer_enriched = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=ranked_data, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=False, 
        include_interactors=False)
    
    # Record the cancer type and rename the pathway id column
    cancer_enriched = cancer_enriched.\
        assign(cancer_type=cancer_type).\
        rename(columns={"stId": "pathway_id"})
    
    # Append to the overall dataframe
    all_enrichments = all_enrichments.append(cancer_enriched)

In [6]:
all_enrichments.head()

,pathway_id,name,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total,cancer_type
0,R-HSA-3108214,SUMOylation of DNA damage response and repair ...,0.005581,0.045443,0.84264,81,81,ccrcc
1,R-HSA-4551638,SUMOylation of chromatin organization proteins,0.004272,0.071045,0.84264,62,62,ccrcc
2,R-HSA-4570464,SUMOylation of RNA binding proteins,0.003514,0.093002,0.84264,51,51,ccrcc
3,R-HSA-4615885,SUMOylation of DNA replication proteins,0.003445,0.095357,0.84264,50,50,ccrcc
4,R-HSA-159234,Transport of Mature mRNAs Derived from Intronl...,0.003238,0.102846,0.84264,47,47,ccrcc


## Look at most enriched pathways in each cancer type

In [ ]:
# Boxplot of proteins up/down in each pathway? Sorted with all up together and all down together
# No, not a boxplot. Just binarize up/down. What plot is best to show that?
# Then also make a Reactome overlaid diagram of the pathway

In [16]:
# Save the results
urls_df.to_csv(INDIVIDUAL_ENRICHMENTS_FILE_PATH, sep='\t')